In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
"""
Hybrid MRSO-LNS with Simulated Annealing
Solomon VRPTW Benchmark — Kaggle CSV Format
Dataset: https://www.kaggle.com/datasets/masud7866/solomon-vrptw-benchmark

CSV columns: CUST_NO | XCOORD | YCOORD | DEMAND | READY_TIME | DUE_DATE | SERVICE_TIME
Vehicle info: stored in filename metadata or header rows
"""

import os, math, time, random, glob
import numpy as np
import pandas as pd
from dataclasses import dataclass, field
from typing import List, Tuple, Dict, Optional
from collections import defaultdict

# ═══════════════════════════════════════════════════════════
#  KNOWN SOLOMON VEHICLE PARAMETERS (per category)
# ═══════════════════════════════════════════════════════════

SOLOMON_VEHICLE_PARAMS = {
    "C1":  {"n_vehicles": 10,  "capacity": 200},
    "C2":  {"n_vehicles": 3,   "capacity": 700},
    "R1":  {"n_vehicles": 25,  "capacity": 200},
    "R2":  {"n_vehicles": 25,  "capacity": 1000},
    "RC1": {"n_vehicles": 25,  "capacity": 200},
    "RC2": {"n_vehicles": 25,  "capacity": 1000},
}

# Column aliases — handles both naming styles on Kaggle
COL_ALIASES = {
    "CUST_NO":       ["CUST_NO", "CUST NO.", "CUST NO", "ID", "NODE", "CUSTOMER"],
    "XCOORD":        ["XCOORD", "XCOORD.", "X", "X_COORD", "X COORD"],
    "YCOORD":        ["YCOORD", "YCOORD.", "Y", "Y_COORD", "Y COORD"],
    "DEMAND":        ["DEMAND"],
    "READY_TIME":    ["READY_TIME", "READY TIME", "READY_TIME_1", "READYTIME", "EARLIEST"],
    "DUE_DATE":      ["DUE_DATE", "DUE DATE", "DUE_TIME", "DUE_TIME_1", "DUETIME", "LATEST"],
    "SERVICE_TIME":  ["SERVICE_TIME", "SERVICE TIME", "SERVICE", "SERVICETIME"],
}


# ═══════════════════════════════════════════════════════════
#  DATA STRUCTURES
# ═══════════════════════════════════════════════════════════

@dataclass
class Customer:
    id: int
    x: float; y: float; demand: float
    ready: float; due: float; service: float

@dataclass
class Problem:
    name: str
    category: str          # e.g. "RC1", "RC2"
    n_vehicles: int
    capacity: float
    depot: Customer
    customers: List[Customer]

    def __post_init__(self):
        self._dist: Dict[Tuple[int,int], float] = {}

    def dist(self, a: Customer, b: Customer) -> float:
        k = (a.id, b.id)
        if k not in self._dist:
            self._dist[k] = math.hypot(a.x - b.x, a.y - b.y)
        return self._dist[k]

@dataclass
class Route:
    stops: List[Customer] = field(default_factory=list)
    def copy(self): return Route(stops=list(self.stops))

@dataclass
class Solution:
    routes: List[Route] = field(default_factory=list)
    total_cost: float = 0.0
    def copy(self):
        s = Solution(routes=[r.copy() for r in self.routes])
        s.total_cost = self.total_cost
        return s


# ═══════════════════════════════════════════════════════════
#  SMART PARSER — handles all Solomon CSV variants
# ═══════════════════════════════════════════════════════════

def _detect_category(name: str) -> str:
    """Infer RC1/RC2/R1/... from filename."""
    n = name.upper()
    for cat in ["RC2", "RC1", "C2", "C1", "R2", "R1"]:
        if n.startswith(cat):
            return cat
    return "RC1"   # fallback

def _resolve_col(df_cols: List[str], field: str) -> Optional[str]:
    """Find which DataFrame column corresponds to a logical field."""
    up = [c.upper().strip() for c in df_cols]
    for alias in COL_ALIASES[field]:
        if alias.upper() in up:
            return df_cols[up.index(alias.upper())]
    return None

def _vehicle_info_from_header(lines: List[str]) -> Tuple[Optional[int], Optional[float]]:
    """
    Try to extract vehicle NUMBER and CAPACITY from raw text lines.
    Handles both:
      - 'VEHICLE\\nNUMBER  CAPACITY\\n25  200'
      - '25,200' style CSV rows
    """
    for i, line in enumerate(lines):
        lo = line.upper()
        if "NUMBER" in lo and "CAPACITY" in lo:
            # next non-empty line has the values
            for j in range(i+1, min(i+4, len(lines))):
                parts = lines[j].replace(",", " ").split()
                if len(parts) >= 2:
                    try:
                        return int(parts[0]), float(parts[1])
                    except ValueError:
                        pass
        # Pattern: first numeric row after header with exactly 2 integers
        if i < 8:
            parts = line.replace(",", " ").split()
            if len(parts) == 2:
                try:
                    n, q = int(parts[0]), float(parts[1])
                    if 1 <= n <= 200 and q > 0:
                        return n, q
                except ValueError:
                    pass
    return None, None

def parse_solomon_csv(filepath: str) -> Problem:
    """
    Universal parser supporting:
    1. Pure CSV with header row (Kaggle format)
    2. Solomon text format embedded in CSV-ish file
    3. Standard Solomon .txt format
    """
    basename = os.path.splitext(os.path.basename(filepath))[0]
    category = _detect_category(basename)
    defaults = SOLOMON_VEHICLE_PARAMS[category]

    with open(filepath, "r", encoding="utf-8-sig") as f:
        raw = f.read()

    lines = [l.strip() for l in raw.splitlines() if l.strip()]

    # ── Try to extract vehicle info from header block ──
    n_veh, cap = _vehicle_info_from_header(lines)
    if n_veh is None:
        n_veh, cap = defaults["n_vehicles"], defaults["capacity"]

    # ── Find the data block (header row with coordinate columns) ──
    data_start = None
    for i, line in enumerate(lines):
        up = line.upper().replace(",", " ")
        if ("XCOORD" in up or "X COORD" in up or " X " in up) and \
           ("YCOORD" in up or "Y COORD" in up or " Y " in up):
            data_start = i
            break

    if data_start is None:
        # Fallback: skip non-numeric header lines, look for 7-column numeric rows
        for i, line in enumerate(lines):
            parts = line.replace(",", " ").split()
            if len(parts) >= 7:
                try:
                    [float(p) for p in parts[:7]]
                    # Assume Solomon text format: id x y d rt dd st
                    data_start = i - 1   # header assumed one line before
                    break
                except ValueError:
                    pass

    if data_start is None:
        raise ValueError(f"Cannot locate data block in {filepath}")

    # ── Parse with pandas ──
    # Reconstruct CSV-like block from data_start onward
    block = "\n".join(lines[data_start:])
    sep = "," if "," in lines[data_start] else r"\s+"

    try:
        df = pd.read_csv(
            __import__("io").StringIO(block),
            sep=sep, engine="python",
            on_bad_lines="skip"
        )
    except Exception:
        df = pd.read_csv(
            __import__("io").StringIO(block),
            sep=r"\s+", engine="python",
            on_bad_lines="skip"
        )

    df.columns = [str(c).strip() for c in df.columns]

    # ── Map columns ──
    col = {}
    for field in COL_ALIASES:
        found = _resolve_col(list(df.columns), field)
        if found:
            col[field] = found

    # Fallback: positional mapping (standard Solomon column order)
    pos_map = ["CUST_NO", "XCOORD", "YCOORD", "DEMAND",
               "READY_TIME", "DUE_DATE", "SERVICE_TIME"]
    if "XCOORD" not in col and len(df.columns) >= 7:
        for i, name in enumerate(pos_map):
            col[name] = df.columns[i]

    required = ["XCOORD", "YCOORD", "DEMAND", "READY_TIME", "DUE_DATE", "SERVICE_TIME"]
    missing = [f for f in required if f not in col]
    if missing:
        raise ValueError(f"Missing columns {missing} in {filepath}. "
                         f"Found: {list(df.columns)}")

    # ── Drop non-numeric rows ──
    df = df[pd.to_numeric(df[col["XCOORD"]], errors="coerce").notna()].copy()
    df = df.reset_index(drop=True)

    customers = []
    for idx, row in df.iterrows():
        cid = int(float(row[col["CUST_NO"]])) if "CUST_NO" in col else idx
        customers.append(Customer(
            id    = cid,
            x     = float(row[col["XCOORD"]]),
            y     = float(row[col["YCOORD"]]),
            demand= float(row[col["DEMAND"]]),
            ready = float(row[col["READY_TIME"]]),
            due   = float(row[col["DUE_DATE"]]),
            service=float(row[col["SERVICE_TIME"]]),
        ))

    if not customers:
        raise ValueError(f"No data rows parsed from {filepath}")

    depot = customers[0]
    depot.id = 0
    return Problem(
        name=basename, category=category,
        n_vehicles=n_veh, capacity=cap,
        depot=depot, customers=customers[1:]
    )


# ═══════════════════════════════════════════════════════════
#  FEASIBILITY & COST
# ═══════════════════════════════════════════════════════════

def route_cost(p: Problem, r: Route) -> float:
    if not r.stops: return 0.0
    c = p.dist(p.depot, r.stops[0])
    for a, b in zip(r.stops, r.stops[1:]): c += p.dist(a, b)
    return c + p.dist(r.stops[-1], p.depot)

def route_feasible(p: Problem, r: Route) -> bool:
    if not r.stops: return True
    if sum(c.demand for c in r.stops) > p.capacity: return False
    t, prev = 0.0, p.depot
    for c in r.stops:
        t += p.dist(prev, c)
        if t > c.due: return False
        t = max(t, c.ready) + c.service
        prev = c
    return t + p.dist(prev, p.depot) <= p.depot.due

def solution_cost(p: Problem, s: Solution) -> float:
    return sum(route_cost(p, r) for r in s.routes if r.stops)

def solution_feasible(p: Problem, s: Solution) -> bool:
    # ① Mỗi route phải thỏa capacity + time window
    if not all(route_feasible(p, r) for r in s.routes):
        return False
    # ② Mỗi khách hàng phải xuất hiện đúng 1 lần (không thiếu, không trùng)
    visited = [c.id for r in s.routes for c in r.stops]
    all_ids = [c.id for c in p.customers]
    return sorted(visited) == sorted(all_ids)

def check_coverage(p: Problem, s: Solution) -> Tuple[List[int], List[int]]:
    """Trả về (missing_ids, duplicate_ids) để debug."""
    from collections import Counter
    cnt     = Counter(c.id for r in s.routes for c in r.stops)
    all_ids = {c.id for c in p.customers}
    missing    = sorted(all_ids - cnt.keys())
    duplicates = sorted(cid for cid, n in cnt.items() if n > 1)
    return missing, duplicates

def get_feasible_customers(p, unvisited, cur, t_cur, q_cur):
    out = []
    for c in unvisited:
        if c.demand > q_cur: continue
        ta = t_cur + p.dist(cur, c)
        if ta > c.due: continue
        td = max(ta, c.ready) + c.service
        if td + p.dist(c, p.depot) > p.depot.due: continue
        out.append(c)
    return out


# ═══════════════════════════════════════════════════════════
#  ACTOR-CRITIC (MRSO)
# ═══════════════════════════════════════════════════════════

class ActorCritic:
    def __init__(self, lr_a=0.005, lr_c=0.005, gamma=0.95):
        self.lr_a = lr_a; self.lr_c = lr_c; self.gamma = gamma
        self.w_a = np.zeros(6); self.w_c = np.zeros(4)
        self._e = 1e-8

    def _feats_action(self, p, cur, cands):
        fs = []
        for c in cands:
            d = p.dist(cur, c)
            fs.append([-d, -p.dist(c, p.depot),
                        max(0.0, c.due - c.ready), c.demand,
                        d / (p.dist(p.depot, c) + self._e), 1.0])
        return np.array(fs, dtype=float)

    def _feats_state(self, n_f, avg_d, q, t):
        return np.array([n_f/100., avg_d/1000., q/1000., t/1000.])

    def probs(self, p, cur, cands):
        f = self._feats_action(p, cur, cands)
        lg = f @ self.w_a; lg -= lg.max()
        pr = np.exp(lg); return pr / (pr.sum() + self._e)

    def select(self, pr): return int(np.random.choice(len(pr), p=pr))

    def train(self, exps, p):
        G, rets = 0.0, []
        for e in reversed(exps): G = e[3] + self.gamma * G; rets.insert(0, G)
        for i, (si, cands, ai, _, _) in enumerate(exps):
            if not cands: continue
            cur, n_f, avg_d, q, t = si
            sf = self._feats_state(n_f, avg_d, q, t)
            td = rets[i] - float(self.w_c @ sf)
            self.w_c += self.lr_c * td * sf
            f = self._feats_action(p, cur, cands)
            pr = self.probs(p, cur, cands)
            g = f[ai] - (pr[:, None] * f).sum(0)
            self.w_a += self.lr_a * td * g


# ═══════════════════════════════════════════════════════════
#  ALGORITHM 1 — MRSO Construct Solution
# ═══════════════════════════════════════════════════════════

def mrso_construct(agent, p, epsilon):
    s, unvisited, exps = Solution(), list(p.customers), []
    while unvisited:
        r, cur, t, q = Route(), p.depot, 0.0, p.capacity
        while True:
            feas = get_feasible_customers(p, unvisited, cur, t, q)
            if not feas: break
            n_f = len(feas)
            avg_d = float(np.mean([p.dist(cur, c) for c in feas]))
            si = (cur, n_f, avg_d, q, p.depot.due - t)
            if random.random() < epsilon:
                c_nxt = random.choice(feas); ai = feas.index(c_nxt)
            else:
                pr = agent.probs(p, cur, feas); ai = agent.select(pr); c_nxt = feas[ai]
            exps.append((si, feas, ai, -p.dist(cur, c_nxt), None))
            r.stops.append(c_nxt); unvisited.remove(c_nxt)
            t = max(t + p.dist(cur, c_nxt), c_nxt.ready) + c_nxt.service
            q -= c_nxt.demand; cur = c_nxt
        if r.stops: s.routes.append(r)
    s.total_cost = solution_cost(p, s)
    return s, exps


# ═══════════════════════════════════════════════════════════
#  ALGORITHM 2 — LNS Search
# ═══════════════════════════════════════════════════════════

def _random_remove(s, n, p):
    s2 = s.copy()
    all_c = [c for r in s2.routes for c in r.stops]
    if not all_c: return s2, []
    rem = random.sample(all_c, min(n, len(all_c)))
    for r in s2.routes: r.stops = [c for c in r.stops if c not in rem]
    s2.routes = [r for r in s2.routes if r.stops]
    return s2, rem

def _worst_remove(s, n, p):
    s2 = s.copy()
    contribs = []
    for r in s2.routes:
        st = r.stops
        for i, c in enumerate(st):
            prev = st[i-1] if i > 0 else p.depot
            nxt  = st[i+1] if i < len(st)-1 else p.depot
            contribs.append((p.dist(prev,c)+p.dist(c,nxt)-p.dist(prev,nxt), c))
    contribs.sort(key=lambda x: -x[0])
    rem = [c for _,c in contribs[:n]]
    for r in s2.routes: r.stops = [c for c in r.stops if c not in rem]
    s2.routes = [r for r in s2.routes if r.stops]
    return s2, rem

def _greedy_insert(s, removed, p):
    """
    Chèn từng khách hàng bị remove vào vị trí rẻ nhất khả thi.
    Nếu KHÔNG tìm được vị trí nào → mở route mới buộc phải chứa khách đó.
    Khách hàng KHÔNG BAO GIỜ bị bỏ rơi (silent drop).
    """
    s2 = s.copy()
    uninserted = []   # track khách chưa chèn được

    for c in random.sample(removed, len(removed)):
        best, br, bi = float('inf'), None, None

        # Thử chèn vào các route hiện có
        for ri, r in enumerate(s2.routes):
            for i in range(len(r.stops) + 1):
                r.stops.insert(i, c)
                if route_feasible(p, r):
                    cost = route_cost(p, r)
                    if cost < best:
                        best, br, bi = cost, ri, i
                r.stops.pop(i)

        if br is not None:
            s2.routes[br].stops.insert(bi, c)
        else:
            # Thử mở route đơn lẻ
            nr = Route(stops=[c])
            if route_feasible(p, nr):
                s2.routes.append(nr)
            else:
                # Không có route nào khả thi → ghi nhớ để xử lý sau
                uninserted.append(c)

    # Xử lý khách chưa chèn được: thêm vào route vi phạm ít nhất
    # (giữ customer trong solution, đánh dấu infeasible qua solution_feasible)
    for c in uninserted:
        nr = Route(stops=[c])
        s2.routes.append(nr)   # route này sẽ bị route_feasible detect là vi phạm

    s2.total_cost = solution_cost(p, s2)
    return s2

def lns_search(s_cur, p, n_lns=30, T0=None, alpha=0.95, n_remove=None):
    if n_remove is None: n_remove = max(3, len(p.customers)//10)
    if T0 is None: T0 = max(1.0, 0.05 * s_cur.total_cost)
    best, T = s_cur.copy(), T0
    for _ in range(n_lns):
        op = random.choice([_random_remove, _worst_remove])
        sp, rem = op(s_cur, n_remove, p)
        sn = _greedy_insert(sp, rem, p)
        if not solution_feasible(p, sn): continue
        delta = sn.total_cost - s_cur.total_cost
        if delta < 0 or random.random() < math.exp(-delta/(T+1e-9)):
            s_cur = sn
            if s_cur.total_cost < best.total_cost: best = s_cur.copy()
        T *= alpha
    return best


# ═══════════════════════════════════════════════════════════
#  ALGORITHM 3★ — Hybrid MRSO-LNS with Simulated Annealing
# ═══════════════════════════════════════════════════════════

def hybrid_mrso_lns_sa(
    p: Problem,
    n_train: int    = 40,
    n_iter: int     = 150,
    p_lns: float    = 0.7,
    T0: float       = None,
    alpha_sa: float = 0.995,
    T_min: float    = None,
    n_lns: int      = 30,
) -> Tuple[Solution, Dict]:

    # ── Phase 1: MRSO Training ──────────────────────────────
    agent = ActorCritic()
    for ep in range(n_train):
        eps = max(0.05, 0.3 * (1 - ep / n_train))
        s_ep, exps = mrso_construct(agent, p, eps)
        agent.train(exps, p)

    # ── Phase 2: Initialize ─────────────────────────────────
    s_best, _ = mrso_construct(agent, p, 0.1)
    s_best.total_cost = solution_cost(p, s_best)
    s_cur = s_best.copy()

    if T0   is None: T0   = 0.1 * s_best.total_cost
    if T_min is None: T_min = 1e-4 * T0

    T = T0  # Line 9★
    stats = {"accepted_worse": 0, "reheats": 0}

    for _ in range(n_iter):                                      # Line 10
        r = random.random()                                      # Line 11
        if r < p_lns:                                            # Line 12-13
            s_new = lns_search(s_cur, p, n_lns=n_lns, T0=T*0.5)
        else:                                                    # Line 15-16
            s_new, _ = mrso_construct(agent, p, 0.05)
            s_new.total_cost = solution_cost(p, s_new)

        # Line 17-20★: SA acceptance
        if solution_feasible(p, s_new):
            delta = s_new.total_cost - s_cur.total_cost         # Line 17★
            if delta < 0 or random.random() < math.exp(-delta/(T+1e-9)):  # 19★
                s_cur = s_new                                    # Line 20★
                if delta > 0: stats["accepted_worse"] += 1
                if s_cur.total_cost < s_best.total_cost:        # Line 21-22
                    s_best = s_cur.copy()

        T *= alpha_sa                                            # Line 25★
        if T < T_min:                                            # Line 26★ Reheating
            T = T0 * 0.5
            stats["reheats"] += 1

    return s_best, stats


# ═══════════════════════════════════════════════════════════
#  BENCHMARK RUNNER
# ═══════════════════════════════════════════════════════════

def count_vehicles(s): return sum(1 for r in s.routes if r.stops)

def avg_route_dist(p, s):
    active = [r for r in s.routes if r.stops]
    return sum(route_cost(p, r) for r in active) / len(active) if active else 0.0

def find_files(data_dir: str, category: str) -> List[str]:
    """Recursively find all files (csv/txt) matching the category prefix."""
    exts = ["*.csv", "*.txt", "*.CSV", "*.TXT", "*"]
    found = []
    for ext in exts:
        pattern = os.path.join(data_dir, "**", f"{category}*{ext.lstrip('*')}")
        found += glob.glob(pattern, recursive=True)
        pattern2 = os.path.join(data_dir, f"{category}*{ext.lstrip('*')}")
        found += glob.glob(pattern2)
    # deduplicate, keep files only
    seen = set()
    result = []
    for f in found:
        if os.path.isfile(f) and f not in seen:
            seen.add(f); result.append(f)
    return sorted(result)

def run_benchmark(
    data_dir: str,
    categories: List[str]  = ["RC1", "RC2"],
    runs: int              = 3,
    n_train: int           = 40,
    n_iter: int            = 150,
    verbose: bool          = True,
) -> Tuple[Dict, pd.DataFrame]:

    all_rows = []
    summary  = {}

    for cat in categories:
        files = find_files(data_dir, cat)
        if not files:
            print(f"[WARN] No files found for {cat} in {data_dir}")
            print(f"       Searched pattern: {cat}*.csv / {cat}*.txt")
            continue

        print(f"\n{'='*62}")
        print(f"  Category : {cat}  |  {len(files)} instances  |  {runs} runs each")
        print(f"{'='*62}")
        print(f"  {'Instance':<12} {'Avg.Dist':>10} {'Veh':>6} "
              f"{'Avg.Cost':>12} {'Time(s)':>9}")
        print(f"  {'-'*54}")

        cat_d, cat_v, cat_c, cat_t = [], [], [], []

        for fpath in files:
            try:
                p = parse_solomon_csv(fpath)
            except Exception as e:
                print(f"  [SKIP] {os.path.basename(fpath)}: {e}")
                continue

            rd, rv, rc_, rt = [], [], [], []
            for run in range(runs):
                t0 = time.time()
                sol, _ = hybrid_mrso_lns_sa(
                    p, n_train=n_train, n_iter=n_iter,
                    p_lns=0.7, alpha_sa=0.995, n_lns=30
                )
                elapsed = time.time() - t0

                # ── Validate coverage ──────────────────────────
                missing, dupes = check_coverage(p, sol)
                if missing:
                    print(f"  [WARN] {os.path.basename(fpath)} run {run+1}: "
                          f"{len(missing)} customers NOT visited: {missing[:5]}...")
                if dupes:
                    print(f"  [WARN] {os.path.basename(fpath)} run {run+1}: "
                          f"{len(dupes)} customers visited >1 time: {dupes[:5]}...")
                # ──────────────────────────────────────────────

                rd.append(avg_route_dist(p, sol))
                rv.append(count_vehicles(sol))
                rc_.append(sol.total_cost)
                rt.append(elapsed)

            md, mv, mc, mt = np.mean(rd), np.mean(rv), np.mean(rc_), np.mean(rt)
            name = os.path.splitext(os.path.basename(fpath))[0]
            if verbose:
                print(f"  {name:<12} {md:>10.2f} {mv:>6.1f} {mc:>12.2f} {mt:>9.1f}")

            cat_d.append(md); cat_v.append(mv)
            cat_c.append(mc); cat_t.append(mt)
            all_rows.append({
                "Category": cat, "Instance": name,
                "Avg.Dist": round(md, 2), "Num.Vehicles": round(mv, 2),
                "Avg.Cost": round(mc, 2), "Time(s)": round(mt, 2),
            })

        if cat_d:
            summary[cat] = {
                "Avg. distance":  round(np.mean(cat_d), 2),
                "Num. vehicles":  round(np.mean(cat_v), 2),
                "Avg. cost":      round(np.mean(cat_c), 2),
                "Avg. time (s)":  round(np.mean(cat_t), 2),
            }

    df = pd.DataFrame(all_rows)
    return summary, df


# ═══════════════════════════════════════════════════════════
#  MAIN
# ═══════════════════════════════════════════════════════════

if __name__ == "__main__":
    DATA_DIR = "/content/drive/My Drive/Colab Notebooks/solomon-benchmark"
    RUNS     = 3
    N_TRAIN  = 40
    N_ITER   = 150

    random.seed(42)
    np.random.seed(42)

    summary, df = run_benchmark(
        data_dir=DATA_DIR,
        categories=["RC1", "RC2"],
        runs=RUNS, n_train=N_TRAIN, n_iter=N_ITER,
        verbose=True,
    )

    # ── Final summary ────────────────────────────────────────
    print(f"\n{'='*62}")
    print("  FINAL SUMMARY — Hybrid MRSO-LNS-SA  (Algorithm 3★)")
    print(f"{'='*62}")
    print(f"  {'Category':<10} {'Avg.Dist':>12} {'Veh':>8} "
          f"{'Avg.Cost':>12} {'Time(s)':>10}")
    print(f"  {'-'*56}")
    for cat, res in summary.items():
        print(f"  {cat:<10} {res['Avg. distance']:>12.2f} "
              f"{res['Num. vehicles']:>8.2f} "
              f"{res['Avg. cost']:>12.2f} "
              f"{res['Avg. time (s)']:>10.2f}")
    print(f"{'='*62}")

    # ── Save CSV ──────────────────────────────────────────────
    out = "results_rc1_rc2.csv"
    df.to_csv(out, index=False)
    print(f"\nSaved to {out}")



  Category : RC1  |  16 instances  |  3 runs each
  Instance       Avg.Dist    Veh     Avg.Cost   Time(s)
  ------------------------------------------------------
  RC101            101.97   17.7      1795.68      17.0
  RC102            100.62   16.3      1643.27      18.4
  RC103            104.42   14.3      1495.72      19.6
  RC104            102.31   12.0      1227.77      19.5
  RC105            100.19   17.3      1735.65      17.7
  RC106            104.07   14.7      1525.58      17.7
  RC107             99.94   14.3      1431.79      19.6
  RC108             99.86   13.0      1296.99      19.7
  RC101            101.08   18.0      1817.17      16.6
  RC102            104.88   15.7      1641.74      18.5
  RC103            105.17   13.7      1435.77      20.0
  RC104            103.81   12.3      1279.98      19.2
  RC105             99.11   17.7      1748.66      17.1
  RC106            100.68   15.3      1542.84      18.5
  RC107            101.72   14.0      1424.06      1